# Day 06 — Viral Content Trend Explorer: analysis

**Fetch date: 2026-11-10.** Two 30-day cohorts of recently published data/AI YouTube videos:
prior window 2026-09-11 → 2026-10-11 (Window B) vs recent window 2026-10-11 → 2026-11-10 (Window A).

> Trends here compare two 30-day cohorts of recently published videos. The API gives a current
> snapshot of each video's statistics, not daily history — so these are cohort comparisons, not a
> record of views rising or falling over time.

**Read this first — the truncation that shapes everything below.** Each of the 10 query-windows
(5 queries × 2 windows) returned exactly 50 videos, the one-page cap, so every window holds only
its newest videos: the recent window effectively covers 2026-11-08 → 11-09 and the prior window
2026-10-08 → 10-10. Weekly buckets W1–W3 and W6–W8 are empty, the holdout backtest cannot be
computed, recent-window velocity is undefined (no recent video is aged ≥ 7 days), and the
5-video sentiment snapshot has no eligible video. Nothing below is re-tuned to hide that: the
thresholds, taxonomy and age guard are fixed by plan.md §4 and a miss is reported, not tuned away.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def _find_root() -> Path:
    """Resolve the project root from this notebook file's own location.

    Walks up from the notebook file's directory looking for
    data/processed/videos.csv. Never resolves from the working directory
    alone (Day 1 lesson e): a copy executed outside the project must fail
    with a clear FileNotFoundError, not silently resolve to <cwd>/data/.
    """
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed" / "videos.csv").exists():
            return candidate
    raise FileNotFoundError(
        "could not find data/processed/videos.csv above the notebook file "
        f"({here}); run this notebook from inside the day-06 project folder"
    )


ROOT = _find_root()
sys.path.insert(0, str(ROOT))

from src.trends import backtest_holdout, build_trend_table, sensitivity_check, weekly_volume_table

PROCESSED = ROOT / "data" / "processed"

videos = pd.read_csv(PROCESSED / "videos.csv", parse_dates=["published_at"])
comments = pd.read_csv(PROCESSED / "comments.csv")
trend_table = pd.read_csv(PROCESSED / "trend_table.csv")
print(f"videos: {len(videos)} rows, {videos['window'].value_counts().to_dict()}")
print(f"comments: {len(comments)} rows (sentiment snapshot)")
print(f"topics: {videos['topic'].value_counts().to_dict()}")
print(f"weeks: {videos['week'].value_counts().sort_index().to_dict()}")

## Q1 — Which topics are accelerating and which are fading, and by how much?

Fixed thresholds (plan.md §4.5): accelerating = share change ≥ +3 pp **and** volume growth ≥ +20%
with n ≥ 10 in both windows; fading is the mirror. Everything below is recomputed from
`videos.csv` through `src/trends.py` — the committed `trend_table.csv` is checked against it.

In [ ]:
recomputed = build_trend_table(videos)
check = recomputed.merge(
    trend_table, on="topic", suffixes=("_recomputed", "_committed")
)
for col in ["n_A", "n_B", "share_change_pp", "volume_growth", "classification"]:
    assert (check[f"{col}_recomputed"] == check[f"{col}_committed"]).all(), col
print("trend_table.csv matches a fresh recomputation from videos.csv")
display(
    trend_table[
        ["topic", "n_A", "n_B", "share_A", "share_B", "share_change_pp",
         "volume_growth", "classification", "mixed_signal"]
    ]
)
print()
print("Accelerating:", trend_table.loc[trend_table["classification"] == "accelerating", "topic"].tolist())
print("Fading:", trend_table.loc[trend_table["classification"] == "fading", "topic"].tolist())
print("Insufficient data:", trend_table.loc[trend_table["classification"] == "insufficient data", "topic"].tolist())

In [ ]:
from IPython.display import Image, display as _display

_display(Image(str(ROOT / "outputs" / "charts" / "01_trend_momentum.png")))

**Reading:** in this sample, AI & Machine Learning gained share (+6.4 pp, +24.2% volume) and SQL
gained share (+3.8 pp, +20.9% volume), so both meet the accelerating rule; Python lost share
(−8.1 pp, −21.6% volume) and meets the fading rule. Career, BI & Visualisation and General Data
Analytics have n < 10 in a window and are reported as insufficient data, not classified. Because
every query-window is truncated to its newest 50 videos, these counts are lower bounds for their
windows — the classification compares two truncated slices (Nov 8–9 vs Oct 8–10), not two full
30-day windows, and must be read that way.

## Q2 — Does the velocity story agree with the volume story, and how stable are the calls?

Velocity is median `views_per_day` (age ≥ 7 days only), always with median age and n. The
age-guard sensitivity check recomputes velocity change with a 14-day guard. The holdout backtest
trains on W1–W7, predicts the top share-growth topic from W1–W3 → W5–W7, and checks top-2 in W8.

In [ ]:
weekly = weekly_volume_table(videos)
display(weekly)
print()
print("Full weeks (W1–W8) with >= 5 videos:",
      int(((weekly.loc[["W1", "W2", "W3", "W4", "W5", "W6", "W7", "W8"]].sum(axis=1)) >= 5).sum()),
      "of 8 (target >= 7 — missed: only W5 clears it)")
display(
    trend_table[
        ["topic", "median_views_per_day_A", "median_views_per_day_B",
         "median_age_A", "median_age_B", "velocity_n_A", "velocity_n_B",
         "velocity_change_7d", "velocity_change_14d", "velocity_sensitive_to_guard"]
    ]
)
print()
print("Age-guard sensitivity:", sensitivity_check(videos))
backtest = backtest_holdout(weekly)
print("Backtest:", backtest)

In [ ]:
from IPython.display import Image, display as _display

_display(Image(str(ROOT / "outputs" / "charts" / "02_weekly_volume.png")))
_display(Image(str(ROOT / "outputs" / "charts" / "03_views_velocity.png")))

**Reading:** the velocity story cannot be checked in this sample — no recent video is aged
≥ 7 days (all 236 were published 2026-11-08/09), so every recent-window velocity median is
undefined, every velocity change is undefined, and no mixed-signal flag can be raised. The
14-day sensitivity check flips nothing, for the same reason. The holdout backtest is **not
computable**: W1–W3 hold no videos, so there is no train growth to predict from, and W8 is empty,
so there is no test week. No hit/miss is claimed. Prior-window velocity medians (for reference
only): Python 5.6/day (median age 30 days, n=88), AI & Machine Learning 4.1/day (31 days, n=62),
SQL 3.6/day (30 days, n=43).

## Q3 — What is the audience mood on the fastest-rising video in each topic?

The mood snapshot is the top video by `views_per_day` per topic in the recent window (age ≥ 7
days, comments enabled), scored with Day 5's lexicon scorer copied verbatim. It describes five
videos at most — never the topics — and no sentiment number enters a trend classification.

In [ ]:
print(f"Sentiment videos: 0 (no eligible video in the recent window)")
print(f"Sentiment comments analysed: {len(comments)}")
if comments.empty:
    print("No mood table to show: every recent video is under 7 days old, "
          "so the snapshot has no video to fetch comments for.")
else:
    display(
        comments.groupby(["video_id", "topic"])
        .agg(mean_compound=("sentiment_score", "mean"), n=("sentiment_score", "size"))
        .reset_index()
    )

In [ ]:
from IPython.display import Image, display as _display

_display(Image(str(ROOT / "outputs" / "charts" / "04_sentiment_mood.png")))

## Long-run context (Layer A — Day 4 frame, relevance sample, not exhaustive)

The Day 4 frame is read-only context, never mixed into the momentum math (different snapshot
date 2026-11-08, different selection method: relevance-ranked).

In [ ]:
day04 = pd.read_csv(
    ROOT.parent / "day-04-youtube-video-performance-analyzer" / "data" / "processed" / "videos.csv"
)
print(f"Day 4 frame: {len(day04)} videos, "
      f"{day04['published_at'].min()[:10]} → {day04['published_at'].max()[:10]}")
print(day04["topic"].value_counts().to_dict())
print(f"Overlap with this sample: {int(videos['video_id'].isin(day04['video_id']).sum())} videos")
print(f"Median engagement rate in this sample: {videos['engagement_rate'].median():.4f} "
      f"(Day 4 sanity range 0.5–8%)")
print(f"Other-topic share: {(videos['topic'] == 'other').mean() * 100:.1f}% (target <= 30%)")
print(f"Shorts share: recent {(videos[videos['window'] == 'A']['is_shorts']).mean() * 100:.1f}%, "
      f"prior {(videos[videos['window'] == 'B']['is_shorts']).mean() * 100:.1f}%")

## Limitations (stated, not hidden)

- **Snapshot, not history.** The API gives a current snapshot of each video's statistics, not
  daily history. Every "trend" here is a comparison of two publish-date cohorts.
- **Truncation.** All 10 query-windows hit the 50-result one-page cap, so the recent window is
  effectively Nov 8–9 and the prior window Oct 8–10. Weekly buckets W1–W3 and W6–W8 are empty;
  volume numbers are lower bounds.
- **Velocity undefined for the recent window.** No recent video is aged ≥ 7 days, so the
  velocity cross-check, the mixed-signal flag and the age-guard sensitivity check all come back
  empty — honestly, not filled in.
- **Backtest not computable.** W1–W3 and W8 are empty.
- **Sentiment snapshot empty.** 0 videos, 0 comments — the same truncation, not a scoring failure.
- **Sentiment is a model output**, not ground truth, and describes five videos at most.
- **No causal claims.** A classification is a statement about this sample only.